### Environment setup
```!pip install openai numpy tqdm tiktoken```

### Import Packages

In [33]:
from openai import OpenAI
import numpy as np
from typing import List, Tuple, Dict
import os
import json
import random
from tqdm import tqdm
import tiktoken
import re

### Utility Functions

In [42]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\$", "", text)
    text = re.sub(r"(?s).*#### ", "", text)
    text = re.sub(r"\.$", "", text)
    text = re.sub(r",", "", text)
    
    if not text:
        return "-1000000000"
    
    return text

def extract_value(text: str) -> str:
    pattern = r"(-?[$0-9.,]{2,})|(-?[0-9]+)"
    matches = re.findall(pattern, text)
    
    if matches:
        for match_groups in matches[::-1]:
            for group in match_groups:
                if group:
                    return clean_text(group)
    
    return "-1000000000"

def load_dataset(file_path: str, sample_size: int = 20) -> List[Dict]:
    """Load and sample from dataset."""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return random.sample(data, sample_size)

def count_tokens(text: str, model: str = "p50k_base") -> int:
    """Count tokens in text using tiktoken."""
    encoding = tiktoken.get_encoding(model)
    return len(encoding.encode(text))

def evaluate_accuracy(predictions: List[str], ground_truth: List[str]) -> float:
    """Calculate accuracy of predictions."""
    correct = 0
    for pred, truth in zip(predictions, ground_truth):
        try:
            if int(pred) == int(truth):
                correct += 1
        except:
            pass
    return correct / len(predictions)

### Chain-of-Thoughts Implementation

1. Visit [DeepInfra](https://deepinfra.com/) and **register an account**. Familiarize yourself with how to use the API by referring to the [documentation](https://deepinfra.com/docs).  

2. Test the **API call** functionality provided by DeepInfra to ensure proper integration.


In [35]:
class ChainOfThought:
    def __init__(self, api_key: str, base_url: str = "https://api.deepinfra.com/v1/openai",
                 model: str = "Qwen/Qwen2.5-7B-Instruct", temperature: float = 0.7):
        self.client = OpenAI(
            api_key=api_key,
            base_url=base_url
        )
        self.model = model
        self.temperature = temperature
        self.total_tokens = 0

    def solve(self, question: str) -> Tuple[str, int]:
        prompt = f"""Please solve this math problem step by step.
Question: {question}
Let's think step by step."""

        try:
            messages = [{"role": "user", "content": prompt}]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=self.temperature,
            )
            tokens_used = count_tokens(prompt) + count_tokens(response.choices[0].message.content)
            self.total_tokens += tokens_used
            return response.choices[0].message.content, tokens_used
        except Exception as e:
            print(f"Error in API call: {e}")
            return "", 0

### Tree-of-Thoughts Implementation

1. You are *highly encouraged* to read the [original paper](http://arxiv.org/abs/2501.02497) and run the [codebase](https://github.com/princeton-nlp/tree-of-thought-llm) first.  
    - Otherwise, you may have no idea what ToT is doing.
    
    - For simplicity, start by running the basic configuration:
        - Search algorithm: BFS

        - Thought generator: propose prompt

        - Task: Game of 24

2. Regarding your own implementation below, feel free to experiment with various hyperparameters, including:
    - API call parameters (e.g., `temperature`)
    
    - ToT implementation parameters (e.g., `max_steps`, `n_samples_per_step`)

3. [Optional] You could try other **7B-level** base models instead of `Qwen2.5-7B-Instruct` in DeepInfra. You might achieve better results with proper implementation.

4. [Optional] Multi-model setups are also allowed—for example, using one model to generate thoughts and another to evaluate them (reward model?).

In [36]:
class TreeOfThoughts:
    def __init__(self, api_key: str, base_url: str = "https://api.deepinfra.com/v1/openai", 
                 model: str = "Qwen/Qwen2.5-7B-Instruct", temperature: float = 0.7):
        """Initialize the Tree-of-Thoughts solver."""
        # TODO: Initialize the OpenAI client and other necessary attributes
        # pass
        

    def chat_with_gpt(self, prompt: str, n: int = 1, stop: str = None) -> List[str]:
        """Get completions from GPT model."""
        # [IMPORTANT] `stop` is important here. You can refer to the implementation of the Game of 24 in the original ToT codebase.
        # TODO: Implement the chat completion function
        # pass
        

    def generate_thoughts(self, question: str, current_thought: str = "", n_samples: int = 3) -> List[str]:
        """Generate multiple possible next steps in reasoning."""
        # [IMPORTANT] A one-shot example can help the model follow your instructions precisely.
        # TODO: Implement thought generation function
        # pass
        
    
    def evaluate_thought(self, question: str, thought: str, cache: bool = True) -> float:
        """Evaluate the likelihood that a thought process leads to the correct answer."""
        # [IMPORTANT] You can use the base model (Qwen2.5-7B-Instruct) for self-evaluation; however, it is not the only option.
        # TODO: Implement thought evaluation function
        # pass
        

    def select_best_thoughts(self, thoughts: List[str], scores: List[float], k: int = 2) -> List[str]:
        """Select the k best thoughts based on their scores."""
        # TODO: Implement thought selection function
        # pass
        

    def solve(self, question: str, max_steps: int = 8, n_samples_per_step: int = 3, 
              k_best_thoughts: int = 2) -> str:
        """Solve a problem using Tree-of-Thoughts reasoning."""
        # TODO: Implement the main solving function
        # pass
        


### Test ToT Using One Simple Example

In [38]:
def test_tot():
    # Set your API key
    api_key = os.getenv("DEEPINFRA_TOKEN", "6CsmsskJ9LlwYPUMXnsy2LX3u3VgfqIi")
    if not api_key:
        print("Please set DEEPINFRA_TOKEN environment variable")
        return

    # Test with a single example
    test_question = "If John has 5 apples and buys 3 more, how many apples does he have?"
    
    tot_solver = TreeOfThoughts(api_key)
    solution = tot_solver.solve(
        question=test_question,
        max_steps=8,
        n_samples_per_step=3,
        k_best_thoughts=2
    )
    
    answer = extract_value(solution)
    print("Solution:", solution)
    print("Extracted answer:", answer)

test_tot()

Solution: Sure, let's solve this step by step:

1. **Initial Count**: John starts with 5 apples.
2. **Apples Bought**: John buys 3 more apples.
3. **Total Count**: To find the total number of apples, we add the apples he bought to the apples he already had.

So, the calculation is:
\[ 5 \text{ apples} + 3 \text{ apples} = 8 \text{ apples} \]

Therefore, John has 8 apples in total.
Certainly! Let's break it down step by step:

1. **Initial Count**: John starts with 5 apples.
2. **Apples Bought**: John buys 3 more apples.
3. **Total Count**: To find the total number of apples, we add the apples he bought to the apples he already had.

So, the calculation is:
\[ 5 \text{ apples} + 3 \text{ apples} = 8 \text{ apples} \]

Therefore, John has 8 apples in total.

To summarize:
- Step 1: John has 5 apples.
- Step 2: John buys 3 more apples.
- Step 3: John's total apples = 5 + 3 = 8.

Final answer: John has 8 apples.
Extracted answer: 8


### Experiment on the Validation Set

- You can use multithreading to accelerate the process.

In [43]:
# 1. load DeepInfra api key
api_key = os.getenv("DEEPINFRA_TOKEN", "6CsmsskJ9LlwYPUMXnsy2LX3u3VgfqIi")

# 2. Load and sample dataset
dataset = load_dataset("dataset/cs5260_val_random300.jsonl", sample_size=300)

# 3. Initialize reasoning solvers
tot_solver = TreeOfThoughts(api_key)
cot_solver = ChainOfThought(api_key)

# 4. process dataset
tot_results, cot_results = [], []
tot_tokens, cot_tokens = 0, 0

print("\nProcessing questions...")
for item in tqdm(dataset):
    question = item["question"]
    true_answer = item["answer"]
    
    # ToT solving
    tot_solution = tot_solver.solve(
        question=question,
        max_steps=8, 
        n_samples_per_step=3, 
        k_best_thoughts=2 
    )
    tot_answer = extract_value(tot_solution)
    tot_results.append({
        "question_id": item["question_id"],
        "predicted": tot_answer,
        "true": true_answer,
    })
    
    # CoT solving
    cot_solution, tokens = cot_solver.solve(question)
    cot_answer = extract_value(cot_solution)
    cot_results.append({
        "question_id": item["question_id"],
        "predicted": cot_answer,
        "true": true_answer,
    })
    cot_tokens += tokens

# Calculate metrics
tot_accuracy = evaluate_accuracy([r["predicted"] for r in tot_results], 
                                [r["true"] for r in tot_results])
cot_accuracy = evaluate_accuracy([r["predicted"] for r in cot_results], 
                                [r["true"] for r in cot_results])

# Print results
print("\n=== Validation Results ===")
print(f"ToT Accuracy: {tot_accuracy:.2%}")
print(f"CoT Accuracy: {cot_accuracy:.2%}")
print(f"ToT Total Tokens: {tot_solver.total_tokens}")
print(f"CoT Total Tokens: {cot_tokens}")


Processing questions...


 72%|███████▏  | 216/300 [2:32:22<59:15, 42.33s/it]   


APIConnectionError: Connection error.

### Submission

- Refer to [here](https://www.kaggle.com/competitions/cs-5260-spring-2025-assignment-1/data) for submission format

In [ ]:
# TODO: Evaluate on test set and build submission file.